In [0]:
import os, sys

# Add the repo's src folder to the path so the aml package can be imported
def _repo_root(start: str) -> str:
    path = os.path.abspath(start)
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, "src", "aml", "config.py")):
            return path
        path = os.path.dirname(path)
    raise RuntimeError("Repo root not found")

sys.path.insert(0, os.path.join(_repo_root(os.getcwd()), "src"))

from aml.config import CONFIG

In [0]:
# Loads the silver transactions table and defines helper functions used throughout the notebook:
# - laundering_summary for grouped laundering rates, and account_tx, which lists each transaction twice, once for the sender, once for the receiver, for account level analysis

from pyspark.sql import functions as F, Window

tx = spark.table(CONFIG.table(CONFIG.silver_schema, "ibm_transactions"))

# Transaction count, laundering count and laundering rate per group
def laundering_summary(df, *group_cols):
    return (
        df.groupBy(*group_cols)
        .agg(
            F.count("*").alias("transactions"),
            F.sum("is_laundering").alias("laundering"),
        )
        .withColumn("laundering_pct", F.round(F.col("laundering") / F.col("transactions") * 100, 4))
    )

# One row per transaction it takes part in, as sender or receiver
account_tx = (
    tx.select(F.col("from_account_id").alias("account_id"), "transaction_id", "timestamp", "is_laundering", F.lit("out").alias("direction"))
    .union(tx.select(F.col("to_account_id").alias("account_id"), "transaction_id", "timestamp", "is_laundering", F.lit("in").alias("direction")))
)

In [0]:
# Shows dataset size, laundering count and rate, date range, and number of distinct accounts and banks

accounts = account_tx.select("account_id").distinct()
banks = tx.select(F.col("from_bank").alias("bank")).union(tx.select(F.col("to_bank").alias("bank"))).distinct()

display(
    tx.agg(
        F.count("*").alias("transactions"),
        F.sum("is_laundering").alias("laundering"), 
        F.round(F.avg("is_laundering") * 100, 4).alias("laundering_pct"),
        F.min("timestamp").alias("first_timestamp"),
        F.min("timestamp").alias("last_timestamp"),
    )
    .crossJoin(accounts.agg(F.count("*").alias("accounts")))
    .crossJoin(banks.agg(F.count("*").alias("banks")))
)


In [0]:
# Counts missing values in every column

display(tx.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in tx.columns]))

In [0]:
# Finds rows that are identical in every column except the id, and how many of them are laundering

value_cols = [c for c in tx.columns if c != "transaction_id"]
dup_groups = tx.groupBy(*value_cols).count().filter("count > 1")
display(
    dup_groups.agg(
        F.count("*").alias("duplicate_groups"),
        F.sum("count").alias("rows_in_duplicate_groups"),
        F.sum(F.col("is_laundering") * F.col("count")).alias("laundering_rows_in_duplicate_groups"),
    )
)

In [0]:
# Counts transactions with a zero amount, and same-currency transactions where the paid and received amounts differ

display(
    tx.agg(
        F.sum(((F.col("amount_paid") == 0) | (F.col("amount_received") == 0)).cast("int")).alias("zero_amount"),
        F.sum((~F.col("is_cross_currency") & (F.col("amount_paid") != F.col("amount_received"))).cast("int")).alias("same_currency_amount_mismatch"),
    )
)

In [0]:
# Calculates the exchange rate implied by each cross-currency transaction and summarises it per currency pair; a wide min-max range means inconsistent conversions between the two currencies

display(
    tx.filter(F.col("is_cross_currency") & (F.col("amount_paid") > 0))
    .withColumn("rate", F.col("amount_received") / F.col("amount_paid"))
    .groupBy("payment_currency", "receiving_currency")
    .agg(
        F.count("*").alias("transactions"),
        F.min("rate").alias("min_rate"),
        F.expr("percentile_approx(rate, 0.5)").alias("median_rate"),
        F.max("rate").alias("max_rate"),
    )
    .orderBy(F.desc("transactions"))
)


In [0]:
# Counts transactions and laundering for each calendar day

daily = laundering_summary(tx.withColumn("date", F.to_date("timestamp")), "date").orderBy("date")
display(daily)

In [0]:
# Laundering rate for each day of the week (Sunday = 1)

display(
    laundering_summary(
        tx.withColumn("dow_num", F.dayofweek("timestamp")).withColumn("dow", F.date_format("timestamp", "E")),
        "dow_num", "dow",
    ).orderBy("dow_num")
)

In [0]:
# Laundering rate for each hour of the day

display(laundering_summary(tx.withColumn("hour", F.hour("timestamp")), "hour").orderBy("hour"))

In [0]:
# Percentage of laundering and of normal transactions falling in each hour, to compare their daily patterns

hourly = tx.withColumn("hour", F.hour("timestamp")).groupBy("is_laundering", "hour").count()
display(
    hourly
    .withColumn("pct_of_class", F.round(F.col("count") / F.sum("count").over(Window.partitionBy("is_laundering")) * 100, 2))
    .orderBy("is_laundering", "hour")
)


In [0]:
# Estimates a USD rate for each currency from transactions paid in that currency and received in US Dollar, adds an amount_usd column to every transaction, and counts transactions that could not be converted

usd_rates = (
    tx.filter((F.col("receiving_currency") == "US Dollar") & (F.col("amount_paid") > 0))
    .groupBy(F.col("payment_currency").alias("currency"))
    .agg(F.expr("percentile_approx(amount_received / amount_paid, 0.5)").alias("usd_per_unit"))
)
display(usd_rates.orderBy("currency"))

tx_usd = (
    tx.join(F.broadcast(usd_rates), tx.payment_currency == usd_rates.currency, "left")
    .withColumn("amount_usd", F.col("amount_paid") * F.col("usd_per_unit"))
    .drop("currency", "usd_per_unit")
)

print(f"Transactions without a USD rate: {tx_usd.filter(F.col('amount_usd').isNull()).count():,}")


In [0]:
# Groups transactions by order of magnitude of the USD amount (1 = tens, 2 = hundreds, ...) and shows the laundering rate of each

display(
    laundering_summary(
        tx_usd.filter(F.col("amount_usd") > 0).withColumn("log10_usd", F.floor(F.log10("amount_usd"))),
        "log10_usd",
    ).orderBy("log10_usd")
)

In [0]:
# Laundering rate for round amounts (multiples of 1000 or 100, whole numbers) compared with amounts that have cents

display(
    laundering_summary(
        tx.withColumn(
            "roundness",
            F.when(F.col("amount_paid") % 1000 == 0, "multiple of 1000")
            .when(F.col("amount_paid") % 100 == 0, "multiple of 100")
            .when(F.col("amount_paid") % 1 == 0, "whole number")
            .otherwise("has cents"),
        ),
        "roundness",
    ).orderBy(F.desc("laundering_pct"))
)

In [0]:
# Laundering rate for amounts just below and just above 10,000 USD, to check for structuring

display(
    laundering_summary(
        tx_usd.withColumn(
            "band",
            F.when(F.col("amount_usd").between(8000, 8999.99), "8k-9k")
            .when(F.col("amount_usd").between(9000, 9999.99), "9k-10k")
            .when(F.col("amount_usd").between(10000, 10999.99), "10k-11k")
            .when(F.col("amount_usd").between(11000, 11999.99), "11k-12k"),
        ).filter(F.col("band").isNotNull()),
        "band",
    ).orderBy("band")
)

In [0]:
# Laundering rate for each payment format, highest first

display(laundering_summary(tx, "payment_format").orderBy(F.desc("laundering_pct")))

In [0]:
# Laundering rate for each payment currency, most used first

display(laundering_summary(tx, "payment_currency").orderBy(F.desc("transactions")))

In [0]:
# Laundering rate for each receiving currency, most used first

display(laundering_summary(tx, "receiving_currency").orderBy(F.desc("transactions")))

In [0]:
# Laundering count and rate for each currency pair, cross-currency transactions only

display(
    laundering_summary(tx.filter("is_cross_currency"), "payment_currency", "receiving_currency")
    .orderBy(F.desc("laundering"))
)

In [0]:
# Laundering rate for each payment format, split by whether the transaction is cross-currency

display(laundering_summary(tx, "payment_format", "is_cross_currency").orderBy("payment_format", "is_cross_currency"))

In [0]:
# Laundering rate for each combination of the cross-currency and self-transfer flags

display(laundering_summary(tx, "is_cross_currency", "is_self_transfer").orderBy("is_cross_currency", "is_self_transfer"))

In [0]:
# Laundering rate for transfers within the same bank compared with transfers between different banks

display(laundering_summary(tx.withColumn("is_intra_bank", F.col("from_bank") == F.col("to_bank")), "is_intra_bank"))

In [0]:
# The 25 sending banks with the most laundering transactions

display(laundering_summary(tx, "from_bank").orderBy(F.desc("laundering")).limit(25))

In [0]:
# Builds one row per account with counts of transactions sent and received, laundering transactions, first and last activity, active hours, number of distinct counterparties, and whether it took part in laundering


account_stats = (
    account_tx.groupBy("account_id")
    .agg(
        F.sum((F.col("direction") == "out").cast("int")).alias("sent"),
        F.sum((F.col("direction") == "in").cast("int")).alias("received"),
        F.sum("is_laundering").alias("laundering_tx"),
        F.count("*").alias("total_tx"),
        F.min("timestamp").alias("first_seen"),
        F.max("timestamp").alias("last_seen"),
    )
    .withColumn("involved_in_laundering", F.col("laundering_tx") > 0)
    .withColumn("active_hours", (F.col("last_seen").cast("long") - F.col("first_seen").cast("long")) / 3600)
)

counterparties = (
    tx.groupBy(F.col("from_account_id").alias("account_id")).agg(F.countDistinct("to_account_id").alias("distinct_receivers"))
    .join(
        tx.groupBy(F.col("to_account_id").alias("account_id")).agg(F.countDistinct("from_account_id").alias("distinct_senders")),
        "account_id", "full",
    )
    .fillna(0)
)

account_stats = account_stats.join(counterparties, "account_id", "left")

In [0]:
# Compares activity, counterparties and active time of laundering-involved accounts with all other accounts

display(
    account_stats.groupBy("involved_in_laundering")
    .agg(
        F.count("*").alias("accounts"),
        F.expr("percentile_approx(sent, array(0.5, 0.9, 0.99))").alias("sent_p50_p90_p99"),
        F.expr("percentile_approx(received, array(0.5, 0.9, 0.99))").alias("received_p50_p90_p99"),
        F.expr("percentile_approx(distinct_receivers, array(0.5, 0.9, 0.99))").alias("receivers_p50_p90_p99"),
        F.expr("percentile_approx(distinct_senders, array(0.5, 0.9, 0.99))").alias("senders_p50_p90_p99"),
        F.expr("percentile_approx(active_hours, array(0.1, 0.5, 0.9))").alias("active_hours_p10_p50_p90"),
    )
)

In [0]:
# Counts accounts that only send, only receive, or do both, split by laundering involvement

display(
    account_stats.withColumn(
        "role",
        F.when((F.col("sent") > 0) & (F.col("received") > 0), "sends and receives")
        .when(F.col("sent") > 0, "only sends")
        .otherwise("only receives"),
    )
    .groupBy("role", "involved_in_laundering")
    .count()
    .orderBy("role", "involved_in_laundering")
)

In [0]:
# For accounts involved in laundering, shows what share of their transactions are laundering and how many accounts do nothing but laundering

involved = account_stats.filter("involved_in_laundering").withColumn("laundering_share", F.col("laundering_tx") / F.col("total_tx"))
display(
    involved.agg(
        F.count("*").alias("involved_accounts"),
        F.sum((F.col("laundering_share") == 1).cast("int")).alias("only_laundering"),
        F.expr("percentile_approx(laundering_share, array(0.1, 0.25, 0.5, 0.75, 0.9))").alias("share_p10_p25_p50_p75_p90"),
    )
)

In [0]:
# Among accounts in laundering transactions, counts those that only send, only receive, or both receive and pass on laundering money

display(
    account_tx.filter("is_laundering = 1")
    .groupBy("account_id")
    .agg(F.collect_set("direction").alias("directions"))
    .withColumn("role", F.when(F.size("directions") == 2, "receives and sends").otherwise(F.element_at("directions", 1)))
    .groupBy("role")
    .count()
)

In [0]:
# The 25 accounts with the most transactions

display(account_stats.orderBy(F.desc("total_tx")).limit(25))

In [0]:
# Summarises sender-receiver pairs: how many pairs there are, transactions per pair, and how many pairs have any laundering or only laundering

pairs = laundering_summary(tx, "from_account_id", "to_account_id")
display(
    pairs.agg(
        F.count("*").alias("pairs"),
        F.expr("percentile_approx(transactions, array(0.5, 0.9, 0.99))").alias("tx_per_pair_p50_p90_p99"),
        F.max("transactions").alias("tx_per_pair_max"),
        F.sum((F.col("laundering") > 0).cast("int")).alias("pairs_with_laundering"),
        F.sum((F.col("laundering") == F.col("transactions")).cast("int")).alias("pairs_only_laundering"),
    )
)

In [0]:
# Compares the laundering rate of the first transaction between two accounts with later repeat transactions

pair_order = Window.partitionBy("from_account_id", "to_account_id").orderBy("timestamp", "transaction_id")
display(
    laundering_summary(
        tx.withColumn("is_first_between_pair", F.row_number().over(pair_order) == 1),
        "is_first_between_pair",
    )
)

In [0]:
# For every incoming transaction, finds how long until the receiving account makes its next outgoing payment,buckets that time (1 hour, 1 day, 1 week, later, never) and compares laundering with normal transactions. Events are ordered newest first, so the preceding rows are the later events.

later_first = Window.partitionBy("account_id").orderBy(F.desc("timestamp"), F.desc("direction")).rowsBetween(Window.unboundedPreceding, -1)

onward = (
    account_tx
    .withColumn("next_out_at", F.min(F.when(F.col("direction") == "out", F.col("timestamp"))).over(later_first))
    .filter(F.col("direction") == "in")
    .withColumn("minutes_to_next_out", (F.col("next_out_at").cast("long") - F.col("timestamp").cast("long")) / 60)
    .withColumn(
        "next_out_within",
        F.when(F.col("minutes_to_next_out").isNull(), "never")
        .when(F.col("minutes_to_next_out") <= 60, "1 hour")
        .when(F.col("minutes_to_next_out") <= 24 * 60, "1 day")
        .when(F.col("minutes_to_next_out") <= 7 * 24 * 60, "1 week")
        .otherwise("later"),
    )
)

display(
    onward.groupBy("is_laundering", "next_out_within")
    .count()
    .withColumn("pct_of_class", F.round(F.col("count") / F.sum("count").over(Window.partitionBy("is_laundering")) * 100, 2))
    .orderBy("is_laundering", "next_out_within")
)

In [0]:
# Running percentage of all transactions and of all laundering reached by the end of each day, used to choose the train, validation and test cut-off dates

cumulative = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
totals = tx.agg(F.count("*").alias("t"), F.sum("is_laundering").alias("l")).first()

display(
    daily
    .withColumn("cum_transactions_pct", F.round(F.sum("transactions").over(cumulative) / totals["t"] * 100, 2))
    .withColumn("cum_laundering_pct", F.round(F.sum("laundering").over(cumulative) / totals["l"] * 100, 2))
)